# Prepare Storage for Custom Document Intelligence Training

Run this notebook one cell at a time to create or reuse the lab storage account, grant the required RBAC permissions, and upload the sample forms used for custom model training.

This notebook uses `DefaultAzureCredential`, so it works with your current Azure sign-in context from VS Code, Azure CLI, or another supported Azure identity source.

## Before You Start

Make sure `workshop/.env` exists. From the repository root, you can create it with `Copy-Item workshop/.env.sample workshop/.env` in PowerShell, then fill in the values for your Azure resources.

Required settings: `DOC_INTELLIGENCE_ENDPOINT` and `STORAGE_RESOURCE_GROUP`. Optional settings: `STORAGE_SUBSCRIPTION_ID`, `STORAGE_LOCATION`, `STORAGE_ACCOUNT_NAME`, `DOC_INTELLIGENCE_RESOURCE_NAME`, and `DOC_INTELLIGENCE_RESOURCE_GROUP`.

Your signed-in Azure account needs permission to create storage accounts, update the Document Intelligence resource identity, and assign roles. Owner or User Access Administrator at the resource group or subscription scope is typically sufficient.

In [1]:
import base64
import json
import os
import random
import re
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request
import uuid
from pathlib import Path

from azure.core.exceptions import HttpResponseError, ResourceExistsError, ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.storage.models import StorageAccountCreateParameters, StorageAccountPropertiesCreateParameters
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

## Define Constants

These values match the Python script. The role ID is Azure's built-in `Storage Blob Data Contributor` role.

In [2]:
CONTAINER_NAME = 'sampleforms'
STORAGE_BLOB_DATA_CONTRIBUTOR = 'Storage Blob Data Contributor'
STORAGE_BLOB_DATA_CONTRIBUTOR_ROLE_ID = 'ba92f5b4-2d11-453d-a403-e96b0029c9fe'
STORAGE_ACCOUNT_NAME_PATTERN = re.compile(r'^[a-z0-9]{3,24}$')
COGNITIVE_SERVICES_API_VERSION = '2023-05-01'

## Find the Workshop Files

This cell locates `workshop/.env` and the `sample-forms` folder whether you start the notebook from the repository root or from the notebook folder.

In [3]:
def find_workshop_dir() -> Path:
    start = Path.cwd().resolve()
    candidates = [start / 'workshop', start]

    for parent in start.parents:
        candidates.extend([parent / 'workshop', parent])

    for candidate in candidates:
        if (candidate / '.env').exists() and (candidate / 'lab-01-document-intelligence').exists():
            return candidate

    raise FileNotFoundError('Could not find workshop/.env. Copy workshop/.env.sample to workshop/.env first.')


workshop_dir = find_workshop_dir()
env_path = workshop_dir / '.env'
sample_forms_dir = workshop_dir / 'lab-01-document-intelligence' / 'custom' / 'sample-forms'

if not sample_forms_dir.exists():
    raise FileNotFoundError(f'Sample forms folder not found: {sample_forms_dir}')

print(f'Workshop directory: {workshop_dir}')
print(f'Environment file: {env_path}')
print(f'Sample forms directory: {sample_forms_dir}')

Workshop directory: C:\Users\algut\repos\alesaez\content-processing-solution\workshop
Environment file: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
Sample forms directory: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-01-document-intelligence\custom\sample-forms


## Add Configuration Helpers

These helpers read values from `workshop/.env`, ignore placeholder values, validate storage account names, and save a generated storage account name back to `.env` for reuse.

In [4]:
def optional_setting(name: str) -> str:
    value = (os.getenv(name) or '').strip()
    return '' if not value or value.startswith('<') else value


def require_setting(name: str) -> str:
    value = optional_setting(name)
    if not value:
        raise ValueError(f'Set {name} in workshop/.env')
    return value


def validate_storage_account_name(account_name: str) -> str:
    if not STORAGE_ACCOUNT_NAME_PATTERN.fullmatch(account_name):
        raise ValueError('STORAGE_ACCOUNT_NAME must be 3 to 24 characters and contain only lowercase letters and numbers.')
    return account_name


def get_storage_account_name() -> str:
    configured_name = optional_setting('STORAGE_ACCOUNT_NAME')
    if configured_name:
        return validate_storage_account_name(configured_name)
    return f'aiforms{random.randint(1, 99999):08d}'


def save_env_setting(env_path: Path, name: str, value: str) -> None:
    lines = env_path.read_text(encoding='utf-8').splitlines() if env_path.exists() else []
    setting = f'{name}={value}'

    for index, line in enumerate(lines):
        if line.startswith(f'{name}='):
            lines[index] = setting
            break
    else:
        if lines and lines[-1]:
            lines.append('')
        lines.append(setting)

    env_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

## Load Configuration

This cell loads `.env`, resolves the subscription, resource group, storage account name, and Document Intelligence resource group. Leave `STORAGE_SUBSCRIPTION_ID` blank to use the current Azure CLI subscription.

In [5]:
def get_current_subscription_id() -> str:
    try:
        result = subprocess.run(
            ['az', 'account', 'show', '--output', 'json'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise ValueError('Set STORAGE_SUBSCRIPTION_ID in workshop/.env, or install Azure CLI and run az login.') from exc
    except subprocess.CalledProcessError as exc:
        message = exc.stderr.strip() or exc.stdout.strip()
        raise ValueError(f'Set STORAGE_SUBSCRIPTION_ID in workshop/.env, or run az login. Azure CLI error: {message}') from exc

    account = json.loads(result.stdout)
    subscription_id = (account.get('id') or '').strip()
    if not subscription_id:
        raise ValueError('Set STORAGE_SUBSCRIPTION_ID in workshop/.env, or select a default subscription with az account set.')
    return subscription_id


load_dotenv(env_path)

subscription_id = optional_setting('STORAGE_SUBSCRIPTION_ID') or optional_setting('AZURE_SUBSCRIPTION_ID')
if not subscription_id:
    subscription_id = get_current_subscription_id()

resource_group = require_setting('STORAGE_RESOURCE_GROUP')
doc_intelligence_resource_group = optional_setting('DOC_INTELLIGENCE_RESOURCE_GROUP') or resource_group
location = os.getenv('STORAGE_LOCATION', 'eastus2')
configured_account_name = optional_setting('STORAGE_ACCOUNT_NAME')
account_name = validate_storage_account_name(configured_account_name) if configured_account_name else get_storage_account_name()

print(f'Using subscription: {subscription_id}')
print(f'Storage resource group: {resource_group}')
print(f'Document Intelligence resource group: {doc_intelligence_resource_group}')
print(f'Storage location: {location}')
print(f'Storage account name: {account_name}')

Using subscription: cd244f2d-5f05-4dd1-97aa-d130e20bcd15
Storage resource group: TechExcelAOAI
Document Intelligence resource group: TechExcelAOAI
Storage location: eastus2
Storage account name: aiforms00076132


## Create Azure Clients

The credential object uses your current Azure identity. The management clients handle storage account creation and role assignments.

In [6]:
credential = DefaultAzureCredential()
storage_client = StorageManagementClient(credential, subscription_id)
authorization_client = AuthorizationManagementClient(credential, subscription_id)

print('Azure SDK clients created.')

Azure SDK clients created.


## Create or Reuse the Storage Account

This cell reuses `STORAGE_ACCOUNT_NAME` if it exists. If the value is blank, it creates a new account and saves the generated name back to `workshop/.env`. The account disables public blob access and shared key access.

In [7]:
def get_or_create_storage_account(
    storage_client: StorageManagementClient,
    resource_group: str,
    account_name: str,
    location: str,
):
    try:
        storage_account = storage_client.storage_accounts.get_properties(resource_group, account_name)
        print(f'Using existing storage account: {account_name}')
        return storage_account
    except ResourceNotFoundError:
        pass

    print(f'Creating storage account: {account_name}')
    poller = storage_client.storage_accounts.begin_create(
        resource_group,
        account_name,
        StorageAccountCreateParameters(
            sku={'name': 'Standard_LRS'},
            kind='StorageV2',
            location=location,
            properties=StorageAccountPropertiesCreateParameters(
                allow_blob_public_access=False,
                allow_shared_key_access=False,
                enable_https_traffic_only=True,
                minimum_tls_version='TLS1_2',
            ),
        ),
    )
    return poller.result()


storage_account = get_or_create_storage_account(storage_client, resource_group, account_name, location)
if not configured_account_name:
    save_env_setting(env_path, 'STORAGE_ACCOUNT_NAME', account_name)
    print(f'Saved STORAGE_ACCOUNT_NAME={account_name} to {env_path}')

print(f'Storage account resource ID: {storage_account.id}')

Using existing storage account: aiforms00076132
Storage account resource ID: /subscriptions/cd244f2d-5f05-4dd1-97aa-d130e20bcd15/resourceGroups/TechExcelAOAI/providers/Microsoft.Storage/storageAccounts/aiforms00076132


## Add RBAC Helpers

These helpers assign `Storage Blob Data Contributor` to a principal at the storage account scope. Role assignment names are deterministic so rerunning the notebook is safe.

In [8]:
def get_current_principal_id(credential: DefaultAzureCredential) -> str:
    token = credential.get_token('https://management.azure.com/.default').token
    payload = token.split('.')[1]
    payload += '=' * (-len(payload) % 4)
    claims = json.loads(base64.urlsafe_b64decode(payload.encode('utf-8')))

    principal_id = (claims.get('oid') or '').strip()
    if not principal_id:
        raise ValueError('Could not determine the current Azure principal object ID from the access token.')
    return principal_id


def grant_blob_data_contributor_to_principal(
    authorization_client: AuthorizationManagementClient,
    storage_account_scope: str,
    principal_id: str,
    principal_type: str,
    display_name: str,
) -> None:
    role_definition_id = f'{storage_account_scope}/providers/Microsoft.Authorization/roleDefinitions/{STORAGE_BLOB_DATA_CONTRIBUTOR_ROLE_ID}'
    role_assignment_name = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{storage_account_scope}/{principal_id}/{STORAGE_BLOB_DATA_CONTRIBUTOR_ROLE_ID}'))

    print(f'Granting {display_name} {STORAGE_BLOB_DATA_CONTRIBUTOR} on the storage account...')
    try:
        authorization_client.role_assignments.create(
            storage_account_scope,
            role_assignment_name,
            RoleAssignmentCreateParameters(
                role_definition_id=role_definition_id,
                principal_id=principal_id,
                principal_type=principal_type,
            ),
        )
    except HttpResponseError as exc:
        if exc.error and exc.error.code == 'RoleAssignmentExists':
            print(f'{display_name} already has {STORAGE_BLOB_DATA_CONTRIBUTOR}.')
            return
        raise ValueError(
            f'Could not assign {STORAGE_BLOB_DATA_CONTRIBUTOR}. '
            'Your account must have Owner or User Access Administrator permission on the storage account. '
            f'Azure error: {exc.message}'
        ) from exc

## Grant Your User Blob Data Access

This role lets your signed-in user upload the training files to the storage container through Microsoft Entra authentication.

In [9]:
current_user_principal_id = get_current_principal_id(credential)
grant_blob_data_contributor_to_principal(
    authorization_client,
    storage_account.id,
    current_user_principal_id,
    'User',
    'current user',
)

Granting current user Storage Blob Data Contributor on the storage account...


## Enable Document Intelligence Managed Identity

Document Intelligence Studio needs the Document Intelligence resource identity to read the training files. This cell finds the resource, enables its system-assigned managed identity if needed, and returns the identity principal ID.

In [10]:
def get_doc_intelligence_account_name() -> str:
    configured_name = optional_setting('DOC_INTELLIGENCE_RESOURCE_NAME')
    if configured_name:
        return configured_name

    endpoint = require_setting('DOC_INTELLIGENCE_ENDPOINT')
    host = urllib.parse.urlparse(endpoint).hostname or ''
    account_name = host.split('.', 1)[0]
    if not account_name:
        raise ValueError('Set DOC_INTELLIGENCE_RESOURCE_NAME in workshop/.env, or use a standard Document Intelligence endpoint.')
    return account_name


def arm_request(credential: DefaultAzureCredential, method: str, url: str, body: dict | None = None) -> dict:
    token = credential.get_token('https://management.azure.com/.default').token
    data = json.dumps(body).encode('utf-8') if body is not None else None
    request = urllib.request.Request(
        url,
        data=data,
        method=method,
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
        },
    )

    try:
        with urllib.request.urlopen(request) as response:
            response_body = response.read().decode('utf-8')
    except urllib.error.HTTPError as exc:
        error_body = exc.read().decode('utf-8', errors='replace')
        raise ValueError(f'Azure management request failed: {exc.code} {exc.reason}. {error_body}') from exc

    return json.loads(response_body) if response_body else {}


def get_doc_intelligence_managed_identity_principal_id(
    credential: DefaultAzureCredential,
    subscription_id: str,
    resource_group: str,
) -> str:
    account_name = get_doc_intelligence_account_name()
    resource_id = (
        f'/subscriptions/{subscription_id}/resourceGroups/{resource_group}'
        f'/providers/Microsoft.CognitiveServices/accounts/{account_name}'
    )
    url = f'https://management.azure.com{resource_id}?api-version={COGNITIVE_SERVICES_API_VERSION}'
    account = arm_request(credential, 'GET', url)

    identity = account.get('identity') or {}
    principal_id = (identity.get('principalId') or '').strip()
    if principal_id:
        print(f'Document Intelligence managed identity is already enabled for: {account_name}')
        return principal_id

    print(f'Enabling system-assigned managed identity on Document Intelligence resource: {account_name}')
    account = arm_request(credential, 'PATCH', url, {'identity': {'type': 'SystemAssigned'}})
    principal_id = ((account.get('identity') or {}).get('principalId') or '').strip()
    if not principal_id:
        raise ValueError(
            'Could not enable or read the Document Intelligence managed identity. '
            'Your account must have permission to update the Document Intelligence resource.'
        )
    return principal_id


doc_intelligence_principal_id = get_doc_intelligence_managed_identity_principal_id(
    credential,
    subscription_id,
    doc_intelligence_resource_group,
)
print(f'Document Intelligence managed identity principal ID: {doc_intelligence_principal_id}')

Document Intelligence managed identity is already enabled for: docintelltecjexcelaiai
Document Intelligence managed identity principal ID: e48171d5-63ae-40f0-abd8-a12893fd3f64


## Grant Document Intelligence Blob Data Access

This role assignment lets the Document Intelligence service read the training files in the storage account during custom model training.

In [11]:
grant_blob_data_contributor_to_principal(
    authorization_client,
    storage_account.id,
    doc_intelligence_principal_id,
    'ServicePrincipal',
    'Document Intelligence managed identity',
)

Granting Document Intelligence managed identity Storage Blob Data Contributor on the storage account...


## Add Upload Helpers

These helpers create the `sampleforms` container and upload everything under `custom/sample-forms`. The retry wrapper handles the short delay Azure RBAC sometimes needs before data-plane permissions take effect.

In [12]:
def upload_sample_forms(blob_service_client: BlobServiceClient, sample_forms_dir: Path) -> None:
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    try:
        container_client.create_container()
    except ResourceExistsError:
        pass

    for file_path in sample_forms_dir.rglob('*'):
        if not file_path.is_file():
            continue

        blob_name = file_path.relative_to(sample_forms_dir).as_posix()
        with file_path.open('rb') as data:
            container_client.upload_blob(name=blob_name, data=data, overwrite=True)
        print(f'Uploaded: {blob_name}')


def upload_sample_forms_with_rbac_retry(blob_service_client: BlobServiceClient, sample_forms_dir: Path) -> None:
    for attempt in range(1, 7):
        try:
            upload_sample_forms(blob_service_client, sample_forms_dir)
            return
        except HttpResponseError as exc:
            if exc.error_code != 'AuthorizationPermissionMismatch' or attempt == 6:
                raise
            wait_seconds = attempt * 10
            print(f'Waiting {wait_seconds} seconds for role assignment propagation before retrying upload...')
            time.sleep(wait_seconds)

## Upload the Sample Forms

Run this cell after the RBAC cells complete. If role propagation is still in progress, the cell waits and retries automatically.

In [13]:
account_url = f'https://{account_name}.blob.core.windows.net'
blob_service_client = BlobServiceClient(account_url=account_url, credential=credential)

print('Uploading sample forms...')
upload_sample_forms_with_rbac_retry(blob_service_client, sample_forms_dir)
print('Upload complete.')

Uploading sample forms...
Uploaded: fields.json
Uploaded: Form_1.jpg
Uploaded: Form_1.jpg.labels.json
Uploaded: Form_1.jpg.ocr.json
Uploaded: Form_2.jpg
Uploaded: Form_2.jpg.labels.json
Uploaded: Form_2.jpg.ocr.json
Uploaded: Form_3.jpg
Uploaded: Form_3.jpg.labels.json
Uploaded: Form_3.jpg.ocr.json
Uploaded: Form_4.jpg
Uploaded: Form_4.jpg.labels.json
Uploaded: Form_4.jpg.ocr.json
Uploaded: Form_5.jpg
Uploaded: Form_5.jpg.labels.json
Uploaded: Form_5.jpg.ocr.json
Upload complete.


## Get the Container URL

Use this container URL in Document Intelligence Studio when creating the custom model training dataset.

In [14]:
container_url = f'{account_url}/{CONTAINER_NAME}'

print('-------------------------------------')
print(f'Storage account: {account_name}')
print(f'Container: {CONTAINER_NAME}')
print(f'Container URL: {container_url}')
print('Use the same signed-in Azure account in Document Intelligence Studio to connect to this storage container.')

-------------------------------------
Storage account: aiforms00076132
Container: sampleforms
Container URL: https://aiforms00076132.blob.core.windows.net/sampleforms
Use the same signed-in Azure account in Document Intelligence Studio to connect to this storage container.
